# N063 · 分治的拆分与合并

**目标：** 定义子问题输出并分析合并成本。

**先修：** N038, N062。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 递归式；平衡与不平衡划分；分治正确性；主定理适用条件。

**配套讲解来源：** [同名逐章意见](../../comment/063_divide_and_conquer_patterns.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

上一章（N062）学了"怎么写对递归"，这一章学"什么时候值得递归"——把一个大问题拆成几个小问题、分别解决、再把结果**合并**起来。注意合并不一定像斐波那契那样是"把两个数相加"，本章两道题的合并方式各不相同：

1. **快速幂**：计算 x 的 n 次方。连乘 n 次当然行，但 n=10⁹ 时要乘十亿次。分治的思路是把指数减半：想算 x¹⁰，先算 x⁵ 再平方；想算 x⁵，先算 x² 再平方再补乘一个 x。每层只做一两次乘法，10⁹ 的指数只要约 30 层就见底。
2. **多数元素**：数组里"出现次数超过一半"的数叫严格多数元素。用分治找它：把数组分两半，各自找出候选，再把两个候选合成一个。

具体到数字的小例子：

- `fast_pow(2, 10)` 应返回 1024。为什么 1024？2¹⁰ = (2⁵)² = 32² = 1024；而 2⁵ = (2²)²×2 = 4²×2 = 32。整个过程只做了约 4 次平方加 2 次补乘，比连乘 10 次没省多少——但指数换成 10⁹ 时，连乘是 10⁹ 次，减半法只要约 60 次乘法。
- `majority_divide([2,2,1,1,1,2,2])` 应返回 2。为什么？2 出现 4 次、数组长 7，一半是 3.5，4 > 3.5，所以 2 是严格多数。
- 反例：`majority_divide([1,2,3])` 必须报错而不是返回 1——每个数都只出现 1 次，3//2=1，没有谁超过一半，"多数元素"根本不存在。

第二题还藏着一个重要区分：分治过程返回的叫"候选"，它只保证"如果多数存在，候选就是它"；多数到底存不存在，最后要单独数一遍确认。

## 2. 基础知识：先读懂这些词

- **分治（divide and conquer）**：三步走的解题框架——拆分（把输入分成几份）、求解（对每份递归调用同一个函数）、合并（把子答案组装成总答案）。本章的重点是"合并"花样百出：快速幂的合并是"平方、必要时补乘"；多数元素的合并是"两个候选比人气"。
- **递归式**：用公式描述分治代价的记号，例如多数分治的 T(n)=2T(n/2)+O(n)——两份递归各花 T(n/2)，合并时线性扫一遍花 O(n)。
- **平衡划分 vs 不平衡划分**：把数组对半切叫平衡划分（树高 log n，通常更优）；每次只切掉一块（如快排最坏情况）叫不平衡划分（树高可到 n）。快速幂的"指数减半"是极不平衡中的例外——问题规模每层砍半，层数仍是 log n。
- **指数减半恒等式**：n 为偶数时 xⁿ = (x^(n/2))²；n 为奇数时 xⁿ = (x^(n//2))² × x。这是快速幂递归步的全部数学依据。
- **负指数**：n<0 时 xⁿ = 1/x⁻ⁿ，把负指数转成正指数算倒数。特例是 x=0 且 n<0——0 不能做除数，属于非法输入，必须报错。
- **多数元素与候选验证**：严格多数指出现次数 **大于** n//2（严格超过一半）。分治只产出"候选"，最后必须再全数组数一遍验证它是否真超过半数，不达标就宣布"不存在"。
- **主定理适用条件（概念预告）**：主定理是解递归式的菜单，但前提是子问题规模划分规整、合并代价是多项式级；像快速幂"一个问题分出一个减半子问题"这类形状也能直接逐层求和得 O(log n)，不必硬套菜单。

## 3. 思路推导：从小例子开始

Step 1：写契约。`fast_pow(x, n)`：返回 x 的 n 次方，n 可为负（x=0 且 n<0 非法）。`majority_divide(nums)`：返回 nums 的严格多数元素；不存在则抛 ValueError。

Step 2：快速幂递归步。算 fast_pow(x,n) 时先要 `half = fast_pow(x, n//2)`——只算一次，存起来复用（这是与"连乘"的本质区别）；答案 = half×half，n 为奇数再乘一个 x。

Step 3：负指数换轨。n<0 时检查 x 是否为 0（0 的负次方是 1/0，非法），然后 `return 1/fast_pow(x,-n)`，转入正指数轨道。

Step 4：多数分治。`candidate(lo,hi)` 在区间里找"候选"：区间只剩一个元素时它就是候选；否则切两半各拿一个候选 a、b——a==b 直接定 a；不同则在本区间数一数 a、b 各出现几次，票多者当选（平票取 a）。

Step 5：验证存在。顶层拿到候选后全数组数它的出现次数，`次数 <= len(nums)//2` 就抛 ValueError——分治只担保"若存在则必是它"，存在性要单独验收。

手算演示一（快速幂，notebook 小实例原数据）：n=13 的减半轨迹是：

| 指数 | 子问题指数 | 是否多乘一次 x |
| --- | --- | --- |
| 13 | 6 | 是（13 是奇数） |
| 6 | 3 | 否（6 是偶数） |
| 3 | 1 | 是 |
| 1 | 0 | 是 |

对照算法：x¹³ = (x⁶)²·x，x⁶ = (x³)²，x³ = (x¹)²·x，x¹ = (x⁰)²·x，x⁰=1 是地基。自底向上回代：x¹=x、x³=x²·x、x⁶=x³²、x¹³=x⁶²·x——每层两次乘法（平方 + 奇数层的一次补乘），四层共约 8 次乘法搞定 13 次方。这张表与 notebook 小实例生成的表格一致。再验 `fast_pow(2,10)`：2⁵=32 → 32²=1024，n=10 偶数无需补乘。

手算演示二（多数分治）：`[2,2,1,1,1,2,2]`，下标 0..6。顶层 mid=3，左右各 3、4 个元素。左半 [2,2,1]：mid 切出 [2] 与 [2,1]→[2,1] 内部 [2] 候选 2、[1] 候选 1，数票 2:1 次、1:1 次，平票取 a=2，左半候选 2。右半 [1,1,2,2]：切成 [1]（候选 1）与 [1,2,2]（内部再切 [1] 与 [2,2]，后者候选 2；1 对 2 数票 1:2，候选 2）→ 右半两个候选 1 与 2，数票 1:2 次、2:2 次，平票取 1。顶层候选 a=2、b=1，全区间数票：2 出现 4 次、1 出现 3 次，候选定为 2。最后验证：4 > 7//2=3，返回 2。

## 4. 核心代码：fast_pow

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def fast_pow(x, n):
    if n < 0:
        if x == 0:
            raise ZeroDivisionError('zero cannot have a negative power')
        return 1 / fast_pow(x, -n)
    if n == 0:
        return 1
    half = fast_pow(x, n // 2)
    return half * half * (x if n % 2 else 1)
```

### 逐段读懂代码

### 1. `fast_pow`：指数减半的分治

```python
def fast_pow(x, n):
    if n < 0:
        if x == 0:
            raise ZeroDivisionError('zero cannot have a negative power')
        return 1 / fast_pow(x, -n)
    if n == 0:
        return 1
    half = fast_pow(x, n // 2)
    return half * half * (x if n % 2 else 1)
```

- 第一段处理负指数：`x==0` 且 `n<0` 意味着要算 1/0，抛 ZeroDivisionError 挡住非法输入；否则把问题转成"正指数再取倒数"，`fast_pow(x,-n)` 中 -n 为正，问题换轨后规模就规整了。
- `if n==0: return 1` 是基例：任何数的 0 次方是 1，也是整条减半链的终点（n=1 时 n//2=0，正好落地）。
- `half=fast_pow(x,n//2)` 是全函数的灵魂：**只递归一次**、把结果存进变量。如果写成 `fast_pow(x,n//2)*fast_pow(x,n//2)`，两个调用各算一遍，O(log n) 立刻退化成 O(n)。分治省时的前提是"子问题真的只解一次"。
- 最后一行同时完成"平方 + 按奇偶补乘"：`half*half` 是平方；`x if n%2 else 1` 是三元表达式——n 为奇数（n%2 为 1，真值）乘 x，偶数乘 1（等于不乘）。一个表达式封装了偶数/奇数两条恒等式。

### 2. `majority_divide`：多数元素的分治与验证

```python
def majority_divide(nums):
    if not nums:
        raise ValueError('nonempty input required')

    def candidate(lo, hi):
        if hi - lo == 1:
            return nums[lo]
        mid = (lo + hi) // 2
        a = candidate(lo, mid)
        b = candidate(mid, hi)
        if a == b:
            return a
        ca = sum((nums[i] == a for i in range(lo, hi)))
        cb = sum((nums[i] == b for i in range(lo, hi)))
        return a if ca >= cb else b
    answer = candidate(0, len(nums))
    if sum((x == answer for x in nums)) <= len(nums) // 2:
        raise ValueError('strict majority does not exist')
    return answer
```

- 外层第一行守卫：空数组谈不上多数元素，报错。
- `candidate(lo,hi)` 采用**左闭右开区间**约定：`lo` 含、`hi` 不含。`hi-lo==1` 是"区间只剩一个元素"的基例，直接返回该元素。
- `mid=(lo+hi)//2` 取中点切成 `[lo,mid)` 与 `[mid,hi)`，两半长度至多差 1（平衡划分）；`a`、`b` 是两半各自递归得到的候选。
- `if a==b: return a`：两半候选一致就不用数票了，直接沿用。
- 数票那两行用了一个小技巧：`sum(nums[i]==a for i in range(lo,hi))` 把布尔值 True/False 求和（True 记 1），一行完成"区间内 a 出现几次"。
- `return a if ca>=cb else b`：票多者当候选；平票（ca==cb）取 a，这只是保证结果确定，不影响正确性——反正若多数真的存在，它必然一路赢到最后（原因见第五节）。
- 顶层拿到 `answer` 后，最后一行做存在性验证：全数组数 `answer` 的次数，`<=len(nums)//2` 说明没严格过半，抛 ValueError。注意比较用的是 `<=` 配 `n//2`，即"出现 4 次、长度 7"过关（4>3），"出现 1 次、长度 3"不过关（1<=1）——严格多数的语义由这行兜底。

## 5. 分段实现：按顺序运行

**本章接口：** `fast_pow(x, n)`、`majority_divide(nums)`

### fast_pow

In [1]:
def fast_pow(x, n):
    if n < 0:
        if x == 0:
            raise ZeroDivisionError('zero cannot have a negative power')
        return 1 / fast_pow(x, -n)
    if n == 0:
        return 1
    half = fast_pow(x, n // 2)
    return half * half * (x if n % 2 else 1)

### majority_divide

In [2]:
def majority_divide(nums):
    if not nums:
        raise ValueError('nonempty input required')

    def candidate(lo, hi):
        if hi - lo == 1:
            return nums[lo]
        mid = (lo + hi) // 2
        a = candidate(lo, mid)
        b = candidate(mid, hi)
        if a == b:
            return a
        ca = sum((nums[i] == a for i in range(lo, hi)))
        cb = sum((nums[i] == b for i in range(lo, hi)))
        return a if ca >= cb else b
    answer = candidate(0, len(nums))
    if sum((x == answer for x in nums)) <= len(nums) // 2:
        raise ValueError('strict majority does not exist')
    return answer

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

n=13; rows=[]
while n:
    rows.append((n,n//2,bool(n%2))); n//=2
show_table(['指数','子问题指数','是否多乘一次x'],rows)

指数,子问题指数,是否多乘一次x
13,6,True
6,3,False
3,1,True
1,0,True


## 7. 正确性、适用条件与复杂度

快速幂使用n=2q或2q+1的恒等式；多数分治的候选只需来自两半候选，因为整体严格多数不可能在两半都不是多数。最终计数排除无多数输入。

**代价：** 快速幂O(log|n|)乘法及栈空间；多数分治O(n log n)时间/O(log n)栈空间。

### 展开理解

快速幂为什么对？靠的是那条恒等式：任何正整数 n 要么是偶数，xⁿ=(x^(n/2))²；要么是奇数，n=2q+1，xⁿ=(x^q)²·x。递归在 n//2 上算出子答案，回代时按奇偶套恒等式，得到的必然是 xⁿ。每层 n 至少砍半（n//2 < n 对一切 n≥1 成立），所以有限层后到 0，触发基例返回 1。负指数被第一步换轨到正指数，也不破坏这套论证。

多数分治为什么对？关键一句：**如果一个数在整体里超过一半，它不可能在两半里都输**。用数字说：数组长 7，某数出现 4 次；把它切成 3+4 两半，假如它在左半不是候选、在右半也不是候选，那它在左半至多 1 次（3 个位置的多数至少 2 次）、右半至多 1 次，总共至多 2 次——跟 4 次矛盾。所以"多数必是至少一半的候选"，两个候选再在合并时比一次票，多数总会胜出（平票规则不影响，因为真多数遇到平票意味着两个候选票数相同，可真多数若在其中本应严格更多）。最后的存在性验证把"候选是它"和"它真的存在"分开：`[1,2,3]` 每个数都当过某个小区间的候选，但没人全盘过半，验证一步拦下。

复杂度。快速幂：n 每层砍半，层数约 log₂|n|，每层 2 次乘法，时间 O(log |n|)、递归栈也是 O(log |n|)。感受一下量级：n=10⁹ 时 log₂n≈30，总共约 60 次乘法——一眨眼都嫌多；对比线性连乘要 10⁹ 次，按每秒一亿次乘法也要十秒。多数分治：T(n)=2T(n/2)+O(n)，每层合并数票都是 O(n)，树高 log n，总时间 O(n log n)、栈深 O(log n)。n=10⁵ 时约 10⁵×17≈170 万次比较，瞬间完成；栈深只有约 17 层，离 Python 的递归上限很远。（顺带一提：多数元素有线性的投票法，分治版在这里是为了练"拆分与合并"这个框架本身。）

## 8. 单元测试：每个用例在检查什么

```python
assert fast_pow(2, 10) == 1024
```
正常用例：整指数、偶偶奇混合（10→5→2→1→0），验证减半链条与补乘逻辑。

```python
assert math.isclose(fast_pow(2, -2), 0.25)
```
负指数用例：2⁻²=0.25。用 `math.isclose` 而不是 `==`，因为 `1/fast_pow(2,2)` 走了除法，浮点结果用容差比较更稳妥——这条同时测了"负指数换轨"代码路径。

```python
assert fast_pow(9, 0) == 1
```
边界用例：0 指数是基例本身，任何底的 0 次方都是 1。

```python
assert majority_divide([2, 2, 1, 1, 1, 2, 2]) == 2
```
正常用例：就是第三节手算的数组，2 出现 4 次 > 7//2=3。这个例子还特意让两半候选不同（左 2 右 1），把"合并比票"的路径走全了。

```python
for n in range(1, 8):
    for a in product([0, 1, 2], repeat=n):
        candidate, count = Counter(a).most_common(1)[0]
        if count > n // 2:
            assert majority_divide(a) == candidate
```
穷举对照：长度 1..7、元素只有 0/1/2 的**所有**数组（3+9+27+…+2187=3279 个）全部跑一遍。右侧用 Counter 找出真多数当标准答案；`if count>n//2` 只对"多数确实存在"的输入断言返回值正确。穷举到 7 已经覆盖了各种平票、交替、全同的刁钻形状。

```python
try:
    majority_divide([1, 2, 3])
except ValueError:
    pass
else:
    raise AssertionError('must verify majority existence')
```
异常契约用例：三个互不相同的数没有多数元素，函数必须抛 ValueError 而不是硬给一个答案。`else` 分支专门抓"没抛异常"的失败情形——这验证的正是"候选之后还要验存在"那一步。

In [4]:
import math

assert fast_pow(2, 10) == 1024

assert math.isclose(fast_pow(2, -2), 0.25)

assert fast_pow(9, 0) == 1

assert majority_divide([2, 2, 1, 1, 1, 2, 2]) == 2

from itertools import product

from collections import Counter

for n in range(1, 8):
    for a in product([0, 1, 2], repeat=n):
        candidate, count = Counter(a).most_common(1)[0]
        if count > n // 2:
            assert majority_divide(a) == candidate

try:
    majority_divide([1, 2, 3])
except ValueError:
    pass
else:
    raise AssertionError('must verify majority existence')

print('N063: 所有本章断言通过')

N063: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 比较快速幂和线性乘法。

**练习 2：** 解释0的负指数不在合法输入中。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- 练习 1（比较快速幂和线性乘法）：写一个 `linear_pow(x,n)` 连乘 n 次，再统计两者在 n=10、100、1000、10000 时的乘法次数（可以在函数里放个计数器）。提示：理论比值约是 n/(2·log₂n)——n=1000 时约 50 次对 1000 次；记得说明浮点连乘和反复平方的误差可能略有差异，用 `math.isclose` 验收；边界带上 n=0 和负 n。
- 练习 2（解释 0 的负指数为何不在合法输入中）：从数学上讲 x⁻ᵏ = 1/xᵏ，x=0 时分母为零，未定义；从代码上讲 `1/fast_pow(0,k)` 会抛 ZeroDivisionError。提示：练习内容是"写明输入输出与边界"——合法域是 (x≠0, n 任意) 或 (x 任意, n≥0)，非法点是 (x=0, n<0)；可以设计 `try: fast_pow(0,-1)` 的验收断言，并说明为什么选择抛 ZeroDivisionError 而不是返回 None（让错误类型对上错误成因）。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def fast_pow(x, n):
    if n < 0:
        if x == 0:
            raise ZeroDivisionError('zero cannot have a negative power')
        return 1 / fast_pow(x, -n)
    if n == 0:
        return 1
    half = fast_pow(x, n // 2)
    return half * half * (x if n % 2 else 1)

def majority_divide(nums):
    if not nums:
        raise ValueError('nonempty input required')

    def candidate(lo, hi):
        if hi - lo == 1:
            return nums[lo]
        mid = (lo + hi) // 2
        a = candidate(lo, mid)
        b = candidate(mid, hi)
        if a == b:
            return a
        ca = sum((nums[i] == a for i in range(lo, hi)))
        cb = sum((nums[i] == b for i in range(lo, hi)))
        return a if ca >= cb else b
    answer = candidate(0, len(nums))
    if sum((x == answer for x in nums)) <= len(nums) // 2:
        raise ValueError('strict majority does not exist')
    return answer

# 示例与回归测试
import math

assert fast_pow(2, 10) == 1024

assert math.isclose(fast_pow(2, -2), 0.25)

assert fast_pow(9, 0) == 1

assert majority_divide([2, 2, 1, 1, 1, 2, 2]) == 2

from itertools import product

from collections import Counter

for n in range(1, 8):
    for a in product([0, 1, 2], repeat=n):
        candidate, count = Counter(a).most_common(1)[0]
        if count > n // 2:
            assert majority_divide(a) == candidate

try:
    majority_divide([1, 2, 3])
except ValueError:
    pass
else:
    raise AssertionError('must verify majority existence')

print('N063: 所有本章断言通过')

N063: 所有本章断言通过


## 11. 代表题与迁移

- **50. Pow(x, n)（Pow(x, n)）**：`fast_pow` 的原题，练的是"子问题只解一次 + 指数减半恒等式 + 负指数换轨"这三件事。
- **169. Majority Element（多数元素）**：`majority_divide` 的对照题（题面保证多数必存在，所以那题不用做存在性验证），练的是"两半候选合并 + 全局验证兜底"的分治框架；顺带可对比线性投票法。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。